**AI STORYTELLER**

*   1️⃣ Day 1 : Set up your environment and build your first story from a text
prompt.
*   2️⃣ Day 2 : Turn a single image into a story using AI captioning.
*   3️⃣ Day 3 : Create a multi-image narrative with sequencing and coherence.
*   4️⃣ Day 4 : Edit, style, and export stories with safety controls.
*   5️⃣ Day 5 : Build and showcase your AI Storyteller app with Streamlit

Day 1 — Foundations & Gemini basics

Understand Gemini API usage and authentication.

Write clear prompts and generate single-scene short stories.

Run interactive Colab widgets to input prompts and fetch generations.


---


Day 2 — Image captioning → story

Use BLIP (Hugging Face) to produce descriptive image captions.

Convert captions into robust story prompts for Gemini.

Control tone/length via prompt guidance.


---


Day 3 — Multi-image sequencing & coherence

Batch-process multiple images and produce a chapter outline.

Generate multi-chapter stories with coherent POV, tense, and character consistency.

Techniques to order images (semantic vs. file order) for narrative flow.


---


Day 4 — Editing loop, style control & exports

Perform targeted edits: tone shifts, tightening, and summarization with generation prompts.

Produce audio (TTS) .

Simple filtering strategies.


---


Day 5 — Streamlit app & demo

Build a Streamlit app that accepts prompt and/or images, calls Gemini + BLIP, displays results, and offers exports.

Run a local demo and understand safe ways to share (ngrok caveats, env vars).

Package and present the project.



---



---

Skills & tools mastered

APIs & SDKs: Gemini (Google GenAI SDK), Hugging Face Transformers (BLIP).

Dev tools: Google Colab (interactive notebooks), Streamlit (rapid UI), basic Python packaging.

NLP prompts: Prompt engineering patterns for outline → draft → revise.

Multimodal: Image captioning → narrative generation; ordering/semantics for multi-image stories.

Output formats: PDF export, text download, and basic TTS (gTTS or Gemini TTS).

Security practices: Using environment variables, avoiding hardcoded keys, minimal content filtering.

**DAY 1**

In [ ]:
# @title
%env GEMINI_API_KEY=Paste your own Gemini API key

In [ ]:
!pip install -q transformers pillow google-generativeai

In [ ]:
from google import genai
import os
client=genai.Client()

In [ ]:
if "GEMINI_API_KEY" not in os.environ:
  print("Please set your Gemini API key in the environment variable GEMINI_API_KEY")
else:
  client=genai.Client()
  MODEL="gemini-2.5-flash"

In [ ]:
prompt=input("Enter your Story prompt and press enter:\n")
if prompt.strip()=="":
  print("No prompt entered , Exiting.")
else:
  print(f"Generating story for prompt: {prompt}")
  print("It may take few seconds")
  try:
    resp=client.models.generate_content(model=MODEL,contents=[prompt])
    print("\n----Generated Story----\n")
    print(resp.text)
  except Exception as e:
    print(f"Error occurred while generating story: {e}")

**DAY 2**

In [ ]:
!pip install -q transformers pillow google-generativeai timm

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from google import genai
import os
import io

In [ ]:
if "GEMINI_API_KEY" not in os.environ:
  print("Please set your Gemini API key in the environment variable GEMINI_API_KEY")
else:
  client=genai.Client()
  MODEL="gemini-2.5-flash"

In [ ]:
processor=BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model=BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large")

In [ ]:
from google.colab import files
uploaded=files.upload()

for fn in uploaded.keys():
  image=Image.open(fn).convert('RGB')
  display(image)

In [ ]:
inputs=processor(images=image,return_tensors='pt')
out=model.generate(**inputs)

caption=processor.decode(out[0],
skip_special_tokens=True)

print("Caption generated by BLIP: ")
print(caption)

In [ ]:
story_prompt=(f"Write a Short story(around 500-700 words) based on this scene description: {caption}")
print(story_prompt)

print("Sending this to Gemini. \n")

response = client.models.generate_content(model=MODEL, contents=story_prompt)
story=response.text
print("\n----Generated Story----\n")
print(story)

In [ ]:
with open("generated_story.txt","w")as f:
  f.write(story)

from google.colab import files
files.download("generated_story.txt")

**DAY 3**

In [ ]:
!pip install -q ipywidgets

In [ ]:
from google.colab import files
from PIL import Image
import io

uploaded=files.upload()

images=[]

image_names=[]

for name,file in uploaded.items():
  image=Image.open(io.BytesIO(file)).convert('RGB')
  image_names.append(name)
  images.append(image)
  display(image)

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

processor=BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
blip_model=BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large")

captions=[]

for img in images:
  inputs=processor(images=img,return_tensors='pt')
  out=blip_model.generate(**inputs,max_new_tokens=30)
  caption=processor.decode(out[0],skip_special_tokens=True)
  captions.append(caption)

print("Captions generated from images:")
for i,caption in enumerate(captions):
  print(f"{image_names[i]}: {caption}")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output


tone_dropdown = widgets.Dropdown(
    options=["whimsical", "adventurous", "suspenseful", "romantic", "sci-fi", "mystery"],
    value="adventurous",
    description="Tone:"
)

length_dropdown = widgets.Dropdown(
    options=["Short (100–200 words)", "Medium (200–400 words)", "Long (400–600 words)"],
    value="Medium (200–400 words)",
    description="Length:"
)

generate_button = widgets.Button(description="Generate Story")
output_box = widgets.Output()

display(tone_dropdown, length_dropdown, generate_button, output_box)


In [ ]:
def on_generate_clicked(b):
    with output_box:
        clear_output()

        tone = tone_dropdown.value
        length_map = {
            "Short (100–200 words)": "100–200 words",
            "Medium (200–400 words)": "200–400 words",
            "Long (400–600 words)": "400–600 words"
        }
        length = length_map[length_dropdown.value]

        caption_prompt = "\n".join([f"- {c}" for c in captions])

        outline_prompt = (
            f"Using the following scene descriptions, create a 4-chapter story outline. "
            f"Each chapter should have a title and a short summary.\n\n"
            f"{caption_prompt}\n\nOutline:"
        )

        try:
            outline_response = client.models.generate_content(model=MODEL, contents=outline_prompt)
            outline_text = outline_response.text
            print(" Story Outline:\n")
            print(outline_text)


            full_story = ""
            for i in range(1, 4):
                chapter_prompt = (
                    f"Using the outline below, write Chapter {i} in a {tone} tone. "
                    f"Make it {length}. Add vivid details, good pacing, and consistent characters.\n\n"
                    f"{outline_text}\n\nChapter {i}:"
                )

                chapter_response = client.models.generate_content(model=MODEL, contents=chapter_prompt)
                chapter_text = chapter_response.text
                print(f"\n Chapter {i}:\n")
                print(chapter_text)
                full_story += f"\n\nChapter {i}:\n{chapter_text}"


            with open("multi_image_story.txt", "w") as f:
                f.write(full_story)
            print("\n Story saved as multi_image_story.txt")

            from google.colab import files
            files.download("multi_image_story.txt")

        except Exception as e:
            print(" Error generating story:", e)

generate_button.on_click(on_generate_clicked)


**DAY 4**

In [ ]:
!pip install -q gtts reportlab

In [ ]:
# You can paste your story here or load from file
story_text = """
Chapter 1: The Mysterious Signal
On a humid night in Berhampur, Aarav, a college student deeply fascinated with electronics, picked up a strange, intermittent signal while testing his new IoT receiver. The waves did not match any known frequency—neither police, nor amateur, nor commercial. The signal hinted at patterns, pulses that seemed almost coded. Alone in his room, Aarav scribbled the signal’s details, determined to discover its origin.

Chapter 2: Decoder in the Dark
Unable to sleep, Aarav ran the signal through his homemade decryption algorithm. Each pass revealed more complexity, as fragments converged into a word: “HELP.” Intrigued and unsettled, he wondered whether this was a prank, or a genuine cry from somewhere nearby. He decided to shift his receiver’s direction, following each faint echo deeper into town. The signal grew stronger near the old clock tower, where legends claimed a scientist once vanished without a trace.

Chapter 3: Shadows of the Past
Aarav reached the clock tower minutes before midnight. He found only silence and a worn-out plaque honoring Dr. Mukherjee, the missing scientist. Frustrated but determined, he kept searching until he noticed a loose brick in the wall. Inside was a small device—a prototype transmitter pulsing weakly. As he picked it up, a new burst of code streamed onto his phone: “Room 202, Royal Lodge. Hurry.”

Chapter 4: The Hidden Room
Aarav approached the Royal Lodge, heart pounding. Room 202 was locked, its door sealed with decades-old dust. He bypassed the rusty lock, entering a cramped space filled with electronics—journals, maps, and data chips scattered everywhere. Behind a bookshelf, he discovered a faded diary describing a failed experiment aimed at transmitting thoughts as signals. The last entry ended abruptly: “Code breached, transmission unstable—unknown consequences.”

Chapter 5: Signal of Hope
Rushing to decode the data chips, Aarav found encrypted files—one revealed a short video of Dr. Mukherjee speaking: “If you’re seeing this, my experiment trapped me in the void between signals. Only someone who understands both electronics and empathy can free me.” Aarav realized the transmission needed not just technical skill but genuine compassion. Guided by Dr. Mukherjee’s instructions and his own empathy, Aarav crafted a new signal pattern and sent it back into the ether.

Chapter 6: Shadows Lifted
At dawn, a response arrived—a sequence of familiar melodies, followed by the scientist's voice: “Thank you, Aarav.” Suddenly, all the lights flickered and the final barrier dropped. For the first time in decades, the clock tower bell rang, and stories of impossible signals faded into legend. Aarav, forever changed, knew that sometimes, real magic hides in the code—and in the kindness we offer strangers.
"""

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

def export_pdf(text, filename="story.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    width, height = letter
    text_object = c.beginText(40, height - 40)
    text_object.setFont("Helvetica", 12)

    for line in text.split('\n'):
        for subline in [line[i:i+90] for i in range(0, len(line), 90)]:
            text_object.textLine(subline)
    c.drawText(text_object)
    c.save()

export_pdf(story_text)


from google.colab import files
files.download("story.pdf")

In [ ]:
from gtts import gTTS
from IPython.display import Audio , display
from google.colab import files

voices = {
    "Default English (US Female)": {"lang": "en", "tld": "com"},
    "British Accent": {"lang": "en", "tld": "co.uk"},
    "Australian Accent": {"lang": "en", "tld": "com.au"},
    "Indian Accent": {"lang": "en", "tld": "co.in"},
    "Slow Reading Voice": {"lang": "en", "tld": "com", "slow": True}
}

for label,options in voices.items():
  print(f"Generating Audio: {label}")

  tts=gTTS(
      text=story_text,
      lang=options["lang"],
      tld=options.get("tld","com"),
      slow=options.get("slow",False)

  )

  filename = f"{label.replace(' ', '_').lower()}.mp3"

  tts.save(filename)

  display(Audio(filename=filename,autoplay=False))

  files.download(filename)

**DAY 5**

In [ ]:
# @title
%%writefile app_streamlit_story.py
import streamlit as st #web app framework
from PIL import Image
import io, requests, os
import textwrap
from gtts import gTTS  #translate text to speech
from transformers import BlipProcessor, BlipForConditionalGeneration
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib.utils import ImageReader
from pyngrok import ngrok
import tempfile
import google.generativeai as genai
import torch

#Authencation
NGROK_AUTH_TOKEN = "Pate your own NGROK auth token"
BACKGROUND_IMAGE_URL = "Past ethe image URL"
GEMINI_API_KEY = "Paste your own Gemini API key"

#StreamLit Page Setup/Style
st.set_page_config(page_title="StoryTeller", layout="wide")

st.markdown(
    f"""
    <style>
    .stApp {{
        background-image: url("{BACKGROUND_IMAGE_URL}");
        background-size: cover;
        background-attachment: fixed;
    }}
    section[data-testid="stSidebar"] {{
        background: rgba(0,0,0,0.3);
        backdrop-filter: blur(10px);
        border-radius: 12px;
        padding: 10px;
    }}
    div[data-testid="stFileUploader"] {{
        background: rgba(255,255,255,0.2);
        border-radius: 10px;
        padding: 10px;
    }}
    html, body, h1, h2, h3, h4, h5, h6, p, div, span, label, li, input, textarea {{
        color: #93A8AC !important;
    }}
    .stButton>button, .stDownloadButton>button {{
        color: #93A8AC !important;
        border-color: #93A8AC;
    }}
    </style>
    """,
    unsafe_allow_html=True
)


st.title("Multi-Image AI StoryTeller")
st.markdown("Upload images → Generate story → Export as PDF & MP3")

with st.sidebar:
    tone = st.selectbox("Tone", ["Adventurous", "Whimsical", "Romantic", "Mysterious", "Humorous", "Calm"])
    length_label = st.selectbox("Length", ["Short (200-300 words)", "Medium (300-600 words)", "Long (600-1000 words)"])
    start_ngrok = st.checkbox("Start ngrok tunnel")
    if start_ngrok:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        url = ngrok.connect(8501)
        st.success(f"Public URL: {url}")


uploaded_images = st.file_uploader("Upload multiple images", type=["jpg", "jpeg", "png"], accept_multiple_files=True)

#Caption model
@st.cache_resource
def load_models():
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large").to("cuda" if torch.cuda.is_available() else "cpu")
    return processor, model

processor, blip_model = load_models()

#config gemini
genai.configure(api_key=GEMINI_API_KEY)

@st.cache_resource
def load_gemini_model():
    return genai.GenerativeModel(model_name="models/gemini-2.5-flash")

gemini_model = load_gemini_model()

#captioning the images
def get_captions(images):
    captions = []
    for img in images:
        if img.mode != "RGB":
            img = img.convert("RGB")
        inputs = processor(images=img, return_tensors="pt").to(blip_model.device)
        out = blip_model.generate(**inputs)
        caption = processor.decode(out[0], skip_special_tokens=True)
        captions.append(caption)
    return captions


def generate_story(captions, tone, length_label):
    length_map = {
        "Short (200-300 words)": (200, 300, 800),
        "Medium (300-600 words)": (300, 600, 1200),
        "Long (600-1000 words)": (600, 1000, 1600)
    }
    min_words, max_words, max_tokens = length_map.get(length_label, (300, 600, 1200))

    prompt = (
    f"You are a creative writer. Write a {tone.lower()} story based on the following image captions:\n\n"
    + "\n".join([f"- {cap}" for cap in captions])
    + f"\n\nThe story should be vivid, engaging, and emotionally rich, with a coherent beginning, middle, and end."
    + f"\nMake it approximately between {min_words} and {max_words} words long."
)


    try:
        response = gemini_model.generate_content(
            contents=prompt,
            generation_config=genai.GenerationConfig(
                temperature=0.9,
                top_p=0.95,
                max_output_tokens=max_tokens
            )
        )
        return response.text.strip()
    except Exception as e:
        return f"❌ Error generating story: {e}"

#Pdf generation
def create_pdf(story_text, images):
    buffer = io.BytesIO()
    c = canvas.Canvas(buffer, pagesize=A4)
    w, h = A4

    try:
        bg_img = Image.open(requests.get(BACKGROUND_IMAGE_URL, stream=True).raw).convert("RGB")
        bg = ImageReader(bg_img)
        c.drawImage(bg, 0, 0, width=w, height=h)
    except:
        pass

    c.setFont("Helvetica-Bold", 16)
    c.drawString(50, h - 50, "Generated Story")

    text = textwrap.wrap(story_text, 100)
    y = h - 80
    for line in text:
        if y < 80:
            c.showPage()
            y = h - 80
        c.drawString(50, y, line)
        y -= 15

    if images:
        c.showPage()
        c.setFont("Helvetica-Bold", 16)
        c.drawString(50, h - 50, "Uploaded Images")
        x, y = 50, h - 150
        for img in images:
            img.thumbnail((200, 200))
            c.drawImage(ImageReader(img), x, y, width=img.width, height=img.height)
            x += 220
            if x > w - 200:
                x = 50
                y -= 220
    c.save()
    buffer.seek(0)
    return buffer

#Audio generation
def create_audio(story):
    audio_bytes = io.BytesIO()
    tts = gTTS(story)
    tts.write_to_fp(audio_bytes)
    audio_bytes.seek(0)
    return audio_bytes


#Processing part
if st.button("Generate Story") and uploaded_images:
    pil_images = [Image.open(img) for img in uploaded_images]
    with st.spinner("Generating captions..."):
        captions = get_captions(pil_images)
        for i, cap in enumerate(captions):
            st.write(f"**Image {i+1}**: {cap}")

    with st.spinner("Generating story..."):
        story = generate_story(captions, tone, length_label)
        st.success("Story generated!")
        st.write(story)

    with st.spinner("Creating PDF..."):
        pdf_file = create_pdf(story, pil_images)
        st.download_button("📄 Download Story as PDF", data=pdf_file, file_name="story.pdf", mime="application/pdf")

    with st.spinner("Creating Audio..."):
        audio = create_audio(story)
        st.audio(audio)
        st.download_button("🔊 Download Story as MP3", data=audio, file_name="story.mp3", mime="audio/mpeg")

elif not uploaded_images:
    st.warning("Upload at least one image to begin.")

In [ ]:
ngrok.kill()

In [ ]:
!pip install -q streamlit pyngrok transformers torch gtts reportlab Pillow

!streamlit run app_streamlit_story.py --server.port 8501 &>/content/log.txt &

from pyngrok import ngrok
ngrok.set_auth_token("Paste your NGROK auth token")
url = ngrok.connect(8501)
print("Public URL:", url)